# Twilio OTP Cost & Burst Analysis

Pull Twilio message logs (SMS OTP sends) via the REST API, then hunt the expensive pattern: ForgeRock AM trees that fire multiple OTPs per login — one bug can multiply SMS cost overnight.

> Credentials come from environment variables. Nothing real is committed here.

## 1. Imports — `requests` + `urllib.parse` for paged API pulls, `pandas` + `rich` for analysis.

In [ ]:
import os
import urllib.parse
from datetime import datetime, timedelta

import pandas as pd
import requests
from rich.console import Console

console = Console()
pd.set_option('display.max_columns', 50)

## 2. Configuration
Reads `TWILIO_ACCOUNT_SID` / `TWILIO_AUTH_TOKEN` from the environment — never hardcode these.

In [ ]:
ACCOUNT_SID = os.environ['TWILIO_ACCOUNT_SID']
AUTH_TOKEN  = os.environ['TWILIO_AUTH_TOKEN']

DAYS_BACK   = 7            # analysis window
OTP_SENDER  = ''           # optional: filter to your OTP sender number, e.g. '+15551234567'
BURST_WINDOW_MIN = 5       # same number re-sent inside this window = suspicious
BURST_THRESHOLD  = 3       # ...this many times = likely AM-tree bug

since = (datetime.utcnow() - timedelta(days=DAYS_BACK)).strftime('%Y-%m-%d')
print('analyzing sends since', since)

## 3. Pull message logs (paginated)
Twilio pages at up to 1000 records. We follow `next_page_uri` with `urllib.parse` until the window is exhausted.

In [ ]:
def fetch_messages(account_sid, auth_token, since, sender=''):
    base = f'https://api.twilio.com/2010-04-01/Accounts/{account_sid}/Messages.json'
    params = {'PageSize': 1000, 'DateSent>=': since}
    if sender:
        params['From'] = sender
    url = base + '?' + urllib.parse.urlencode(params)
    out = []
    with requests.Session() as s:
        s.auth = (account_sid, auth_token)
        while url:
            r = s.get(url, timeout=30)
            r.raise_for_status()
            body = r.json()
            out.extend(body.get('messages', []))
            nxt = body.get('next_page_uri')
            url = urllib.parse.urljoin('https://api.twilio.com', nxt) if nxt else None
    return out

raw = fetch_messages(ACCOUNT_SID, AUTH_TOKEN, since, OTP_SENDER)
print(f'{len(raw)} messages pulled')

## 4. Into a DataFrame
Keep the fields that matter for cost + root-cause work.

In [ ]:
df = pd.DataFrame([{
    'sid': m.get('sid'),
    'to': m.get('to'),
    'from': m.get('from'),
    'status': m.get('status'),
    'direction': m.get('direction'),
    'segments': int(m.get('num_segments') or 1),
    'price': float(m.get('price') or 0),
    'price_unit': m.get('price_unit'),
    'sent_at': pd.to_datetime(m.get('date_sent')),
} for m in raw])
df = df.sort_values('sent_at').reset_index(drop=True)
print(df.shape)
df.head()

## 5. Cost overview

In [ ]:
total_cost = df['price'].sum()
by_day = df.assign(day=df['sent_at'].dt.date).groupby('day').agg(
    sends=('sid', 'count'), cost=('price', 'sum')).reset_index()
unit = df['price_unit'].iloc[0] if len(df) else 'USD'
console.print(f'[bold]Total: {len(df)} sends, {total_cost:.2f} {unit} over {DAYS_BACK} days[/bold]')
by_day

## 6. Burst detection — the AM-tree bug signature
One login should mean one OTP. Same destination number getting 3+ OTPs inside 5 minutes is the classic symptom of a ForgeRock AM tree re-entering the OTP node (loop, retry storm, or duplicate journey execution).

In [ ]:
df['prev_sent'] = df.sort_values('sent_at').groupby('to')['sent_at'].shift(1)
df['mins_since_prev'] = (df['sent_at'] - df['prev_sent']).dt.total_seconds() / 60
bursts = df[df['mins_since_prev'] <= BURST_WINDOW_MIN]

burst_counts = (bursts.groupby('to').size().reset_index(name='rapid_resends')
                    .query('rapid_resends >= @BURST_THRESHOLD')
                    .sort_values('rapid_resends', ascending=False))
console.print(f'[bold red]{len(burst_counts)} numbers show burst behavior '
              f'(>={BURST_THRESHOLD} resends within {BURST_WINDOW_MIN} min)[/bold red]')
burst_counts.head(15)

In [ ]:
# what did the bursts cost?
burst_cost = df[df['to'].isin(burst_counts['to'])]['price'].sum()
if total_cost:
    console.print(f'Burst sends cost {burst_cost:.2f} — {100*burst_cost/total_cost:.1f}% of total OTP spend')
else:
    console.print('no spend in window')

## 7. Delivery health & hourly pattern
Failed/undelivered OTPs are a second cost: users retry, trees resend, spend compounds.

In [ ]:
df['status'].value_counts()

In [ ]:
hourly = df.assign(hour=df['sent_at'].dt.floor('h')).groupby('hour').size()
print('peak hour:', hourly.idxmax(), '->', hourly.max(), 'sends')
hourly.tail(24)

## 8. Root-cause read
If bursts cluster on specific hours, check what changed then: AM journey deployments, tree version rollouts, or upstream latency that made users (or the tree) retry. Join `to` against IDM (`userName`/`telephoneNumber`) with the `idm_bulk_extract.ipynb` output to name the affected identities, then fix the tree — the spend stops the same day.